# Pre-processing data for model.

In [2]:
# Basic libraries for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [3]:
# --- 1. SETUP DIRECTORIES ---
PROCESSED_DIR = Path(os.getcwd()).parent / "DATA" / "processed"
CLEANED_DIR = Path(os.getcwd()).parent / "DATA" / "pre_processed_data"

# Create the folders if they don't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DIR.mkdir(parents=True, exist_ok=True) # <-- This creates your new folder!

datasets = ["FD001", "FD002", "FD003", "FD004"]
train, test, rul = {}, {}, {}

# --- 2. LOAD ORIGINAL DATA ---
# This reads from your safe, original folder into your computer's memory
for ds in datasets:
    train[ds] = pd.read_parquet(PROCESSED_DIR / f"train_{ds}.parquet")
    test[ds]  = pd.read_parquet(PROCESSED_DIR / f"test_{ds}.parquet")
    rul[ds]   = pd.read_parquet(PROCESSED_DIR / f"RUL_{ds}.parquet")

print("✅ Data successfully loaded from the original folder!")

# --- 3. MAKE YOUR CHANGES (Drop Columns) ---
# Your original files on the hard drive are perfectly safe. 
# We are only changing the temporary copy in memory right now.

# List the columns you want to drop here:
columns_to_drop = ["op3", "s1", "s5", "s10", "s16", "s19"] 

for ds in datasets:
    # We use errors='ignore' so if a column doesn't exist, it won't crash your code
    train[ds] = train[ds].drop(columns=columns_to_drop, errors='ignore')
    test[ds]  = test[ds].drop(columns=columns_to_drop, errors='ignore')

print("✅ Unwanted columns dropped from memory!")

# --- 4. SAVE TO THE NEW FOLDER ---
# Now we write these modified datasets permanently to the new folder
for ds in datasets:
    train[ds].to_parquet(CLEANED_DIR / f"train_{ds}.parquet", index=False)
    test[ds].to_parquet(CLEANED_DIR / f"test_{ds}.parquet", index=False)
    rul[ds].to_parquet(CLEANED_DIR / f"RUL_{ds}.parquet", index=False)

print(f"✅ Modified datasets securely saved to: {CLEANED_DIR}")

# Quick verify
display(train["FD002"].head())

✅ Data successfully loaded from the original folder!
✅ Unwanted columns dropped from memory!
✅ Modified datasets securely saved to: c:\Users\adnan\OneDrive\Desktop\Projects\NASA_TURBOJET\DATA\pre_processed_data


,unit,cycle,op1,op2,s2,s3,s4,s6,s7,s8,s9,s11,s12,s13,s14,s15,s17,s18,s20,s21
0,1,1,34.9983,0.8400,555.32,1358.61,1137.23,8.00,194.64,2222.65,8341.91,42.02,183.06,2387.72,8048.56,9.3461,334,2223,14.73,8.8071
1,1,2,41.9982,0.8408,549.90,1353.22,1125.78,5.71,138.51,2211.57,8303.96,42.20,130.42,2387.66,8072.30,9.3774,330,2212,10.41,6.2665
2,1,3,24.9988,0.6218,537.31,1256.76,1047.45,9.02,175.71,1915.11,8001.42,36.69,164.22,2028.03,7864.87,10.8941,309,1915,14.08,8.6723
3,1,4,42.0077,0.8416,549.51,1354.03,1126.38,5.71,138.46,2211.58,8303.96,41.96,130.72,2387.61,8068.66,9.3528,329,2212,10.59,6.4701
4,1,5,25.0005,0.6203,537.07,1257.71,1047.93,9.03,175.05,1915.10,7993.23,36.89,164.31,2028.00,7861.23,10.8963,309,1915,14.13,8.5286


In [4]:
# Define the threshold based on the C-MAPSS paper
MAX_RUL = 125

for ds in datasets:
    # Assuming your RUL column is named 'RUL' or 'max_RUL'
    # We use the pandas .clip() function to set an upper limit
    
    # Cap the separate RUL files
    if 'RUL' in rul[ds].columns:
        rul[ds]['RUL'] = rul[ds]['RUL'].clip(upper=MAX_RUL)
        
    

print(f"✅ All RUL values successfully capped at {MAX_RUL}!")

# Quick verify to see the cap in action
display(rul["FD001"].head())

✅ All RUL values successfully capped at 125!


,RUL
0,112
1,98
2,69
3,82
4,91


In [5]:
# Define your parameters
NOISY_SENSOR = 's3'           # Change to 'sensor_3' if your column is spelled differently
WINDOW_SIZE = 10
ENGINE_ID_COL = 'unit' # Standard C-MAPSS column name for the engine ID (change if needed)

for ds in datasets:
    # --- Process Training Data ---
    if NOISY_SENSOR in train[ds].columns and ENGINE_ID_COL in train[ds].columns:
        # Group by engine ID, then apply the rolling mean
        train[ds][NOISY_SENSOR] = (
            train[ds].groupby(ENGINE_ID_COL)[NOISY_SENSOR]
            .transform(lambda x: x.rolling(window=WINDOW_SIZE, min_periods=1).mean())
        )
        
    # --- Process Test Data ---
    if NOISY_SENSOR in test[ds].columns and ENGINE_ID_COL in test[ds].columns:
        test[ds][NOISY_SENSOR] = (
            test[ds].groupby(ENGINE_ID_COL)[NOISY_SENSOR]
            .transform(lambda x: x.rolling(window=WINDOW_SIZE, min_periods=1).mean())
        )

print(f"✅ Rolling mean (window={WINDOW_SIZE}) successfully applied to {NOISY_SENSOR} without crossing engine boundaries!")

# Quick verify to see the smoothed values
display(train["FD001"][[ENGINE_ID_COL, 'cycle', NOISY_SENSOR]].head(15))

✅ Rolling mean (window=10) successfully applied to s3 without crossing engine boundaries!


,unit,cycle,s3
0,1,1,1589.700000
1,1,2,1590.760000
2,1,3,1589.836667
3,1,4,1588.075000
4,1,5,1587.030000
5,1,6,1586.603333
6,1,7,1587.420000
7,1,8,1586.862500
8,1,9,1587.320000
9,1,10,1587.712000


In [6]:
# FD001 and FD003 — single condition, just assign 0
for ds in ["FD001", "FD003"]:
    train[ds]["condition"] = 0
    test[ds]["condition"]  = 0

# FD002 and FD004 — KMeans to find 6 regimes
for ds in ["FD002", "FD004"]:
    scaler = StandardScaler()
    
    # fit on train, apply same to test
    train_scaled = scaler.fit_transform(train[ds][["op1", "op2"]])
    test_scaled  = scaler.transform(test[ds][["op1", "op2"]])
    
    kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
    train[ds]["condition"] = kmeans.fit_predict(train_scaled)
    test[ds]["condition"]  = kmeans.predict(test_scaled)

# verify
for ds in datasets:
    print(f"{ds} — conditions: {sorted(train[ds]['condition'].unique())}")


FD001 — conditions: [np.int64(0)]
FD002 — conditions: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]
FD003 — conditions: [np.int64(0)]
FD004 — conditions: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]


In [7]:
from sklearn.preprocessing import MinMaxScaler

sensor_cols = [col for col in train["FD001"].columns if col.startswith("s")]

# convert sensor columns to float in all datasets first
for ds in datasets:
    train[ds][sensor_cols] = train[ds][sensor_cols].astype(float)
    test[ds][sensor_cols]  = test[ds][sensor_cols].astype(float)

# now normalize per condition
for ds in datasets:
    scaler_dict = {}
    
    # fit and transform train
    for condition in train[ds]["condition"].unique():
        mask = train[ds]["condition"] == condition
        scaler = MinMaxScaler()
        train[ds].loc[mask, sensor_cols] = scaler.fit_transform(train[ds].loc[mask, sensor_cols])
        scaler_dict[condition] = scaler
    
    # transform test using same scalers
    for condition in test[ds]["condition"].unique():
        mask = test[ds]["condition"] == condition
        if condition in scaler_dict:
            test[ds].loc[mask, sensor_cols] = scaler_dict[condition].transform(test[ds].loc[mask, sensor_cols])
    
    print(f"{ds} — normalized!")

# verify
print(train["FD001"][sensor_cols].min().min(), train["FD001"][sensor_cols].max().max())


FD001 — normalized!
FD002 — normalized!
FD003 — normalized!
FD004 — normalized!
0.0 1.0


In [8]:
RUL_CAP = 125

for ds in datasets:
    max_cycle = train[ds].groupby("unit")["cycle"].max().reset_index()
    max_cycle.columns = ["unit", "max_cycle"]
    train[ds] = train[ds].merge(max_cycle, on="unit")
    train[ds]["RUL"] = (train[ds]["max_cycle"] - train[ds]["cycle"]).clip(upper=RUL_CAP)
    train[ds].drop(columns=["max_cycle"], inplace=True)

print("RUL added to all train datasets!")


RUL added to all train datasets!


In [9]:
def create_windows(df, window_size=50):
    X, y, conditions = [], [], []
    
    for unit in df["unit"].unique():
        engine = df[df["unit"] == unit].reset_index(drop=True)
        
        if len(engine) < window_size:
            continue
        
        for i in range(len(engine) - window_size + 1):
            window    = engine.iloc[i : i + window_size][sensor_cols].values
            label     = engine.iloc[i + window_size - 1]["RUL"]
            condition = engine.iloc[i + window_size - 1]["condition"]
            X.append(window)
            y.append(label)
            conditions.append(condition)
    
    return np.array(X), np.array(y), np.array(conditions)


In [11]:
X_list, y_list, cond_list = [], [], []

for ds in datasets:
    X, y, cond = create_windows(train[ds], window_size=50)
    X_list.append(X)
    y_list.append(y)
    cond_list.append(cond)
    print(f"{ds} — X: {X.shape}, y: {y.shape}, conditions: {np.unique(cond)}")

X_train      = np.concatenate(X_list,    axis=0)
y_train      = np.concatenate(y_list,    axis=0)
cond_train   = np.concatenate(cond_list, axis=0)

print(f"\nFinal X_train: {X_train.shape}")
print(f"Final y_train: {y_train.shape}")
print(f"Final cond_train: {cond_train.shape}")


FD001 — X: (15731, 50, 16), y: (15731,), conditions: [0.]
FD002 — X: (41019, 50, 16), y: (41019,), conditions: [0. 1. 2. 3. 4. 5.]
FD003 — X: (19820, 50, 16), y: (19820,), conditions: [0.]
FD004 — X: (49048, 50, 16), y: (49048,), conditions: [0. 1. 2. 3. 4. 5.]

Final X_train: (125618, 50, 16)
Final y_train: (125618,)
Final cond_train: (125618,)


In [12]:
def create_windows_test(df, window_size=50):
    X, conditions = [], []
    
    for unit in df["unit"].unique():
        engine = df[df["unit"] == unit].reset_index(drop=True)
        
        if len(engine) < window_size:
            window = engine[sensor_cols].values
            window = np.pad(window, ((window_size - len(window), 0), (0, 0)), mode='edge')
        else:
            window = engine.iloc[-window_size:][sensor_cols].values
        
        condition = engine.iloc[-1]["condition"]
        X.append(window)
        conditions.append(condition)
    
    return np.array(X), np.array(conditions)


In [13]:
X_test_list, y_test_list, cond_test_list = [], [], []

for ds in datasets:
    X, cond = create_windows_test(test[ds], window_size=50)
    y = rul[ds]["RUL"].values
    X_test_list.append(X)
    y_test_list.append(y)
    cond_test_list.append(cond)
    print(f"{ds} — X: {X.shape}, y: {y.shape}")

X_test    = np.concatenate(X_test_list,    axis=0)
y_test    = np.concatenate(y_test_list,    axis=0)
cond_test = np.concatenate(cond_test_list, axis=0)

print(f"\nFinal X_test:    {X_test.shape}")
print(f"Final y_test:    {y_test.shape}")
print(f"Final cond_test: {cond_test.shape}")


FD001 — X: (100, 50, 16), y: (100,)
FD002 — X: (259, 50, 16), y: (259,)
FD003 — X: (100, 50, 16), y: (100,)
FD004 — X: (248, 50, 16), y: (248,)

Final X_test:    (707, 50, 16)
Final y_test:    (707,)
Final cond_test: (707,)


In [14]:
MODEL_READY_DIR = Path(os.getcwd()).parent / "DATA" / "model_ready"
MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)

np.save(MODEL_READY_DIR / "X_train.npy",    X_train)
np.save(MODEL_READY_DIR / "y_train.npy",    y_train)
np.save(MODEL_READY_DIR / "cond_train.npy", cond_train)
np.save(MODEL_READY_DIR / "X_test.npy",     X_test)
np.save(MODEL_READY_DIR / "y_test.npy",     y_test)
np.save(MODEL_READY_DIR / "cond_test.npy",  cond_test)

print("All arrays saved!")
print(f"X_train:    {X_train.shape}")
print(f"y_train:    {y_train.shape}")
print(f"cond_train: {cond_train.shape}")
print(f"X_test:     {X_test.shape}")
print(f"y_test:     {y_test.shape}")
print(f"cond_test:  {cond_test.shape}")
# Quick verify by loading one of the arrays back
loaded_X_train = np.load(MODEL_READY_DIR / "X_train.npy")
print(f"Loaded X_train: {loaded_X_train.shape}")

All arrays saved!
X_train:    (125618, 50, 16)
y_train:    (125618,)
cond_train: (125618,)
X_test:     (707, 50, 16)
y_test:     (707,)
cond_test:  (707,)
Loaded X_train: (125618, 50, 16)
